# Amazon Store Data — Cleaning & Analysis

## Problem Statement

We have received raw sales/store data exported from **Amazon**. The dataset contains information about products, orders, and customer transactions, but like most real-world data, it comes with quality issues that need to be addressed before any meaningful analysis can be done.

## Goals

1. **Load the Data** — Read the raw JSON data from the source file
2. **Explore the Data** — Understand the structure, columns, and data types
3. **Clean the Data** — Handle missing values, duplicates, incorrect formats, and outliers
4. **Analyse the Data** — Extract insights such as:
   - Top-selling products
   - Revenue trends
   - Customer behaviour patterns
   - Category-wise performance

## Dataset
- **Source:** Amazon Store Export
- **Format:** JSON
- **File:** `store_data.json`

## Step 1: Import Libraries

In [24]:
import json

## Step 2: Load Data

We define a `load_data()` function that reads the JSON file and returns it as a Python list of dictionaries — one dictionary per user/record.

In [25]:
# Load the data from the JSON file
def load_data(filepath):
    with open(filepath, 'r') as file:
        data = json.load(file)
    return data

## Step 3: Explore the Raw Data

Print the raw data to inspect its structure, fields, and any obvious issues like missing values, inconsistent formats, or duplicates.

In [26]:
data = load_data('store_data.json')
print(data)
print(type(data))

[{'name': 'Alice', 'rating': '5 ', 'feedback': 'Great product!!', 'age': '25'}, {'name': 'Bob', 'rating': 'four', 'feedback': 'ok but late Delivery', 'age': '30'}, {'name': ' Charlie', 'rating': 'two', 'feedback': 'BAD EXPERIENCE '}, {'name': 'Diana', 'feedback': 'Loved it!', 'rating': '5', 'age': '28'}, {'name': 'Eve', 'rating': '3.5', 'feedback': 'Average - could be better', 'age': '20'}, {'name': 'Alice', 'rating': '5', 'feedback': 'Great product again!', 'age': '25'}]
<class 'list'>


## Step 4: Clean the Data

The raw data has several issues to fix:
- **Ratings** stored as strings (`'5 '`, `'four'`, `'3.5'`) — convert all to integers
- **Missing fields** — some records lack `age`; fill with `None`
- **Duplicates** — same user appearing multiple times; keep only the first occurrence

In [41]:
# Clean $ structure the data for analysis
def clean_data(data):
    text_to_num = {"one":1, "two":2, "three":3, "four":4, "five":5}
    def is_number(s):
        try:
            float(s)
            return True
        except ValueError:
            return False
    
    clean_data = []
    unique_users = set()
        
    for user in data:
        # Clean rating
        rating = user.get('rating', '')
        if not rating:
            user['rating'] = None
        elif is_number(str(rating).strip()):
            user['rating'] = int(float(str(rating).strip()))
        elif str(rating).strip().lower() in text_to_num:
            user['rating'] = text_to_num[str(rating).strip().lower()]
        else:
            user['rating'] = None
        
        #Handle missing age
        age = user.get('age', None)
        if age is not None and is_number(str(age).strip()):
            user['age'] = int(float(str(age).strip()))
        else:
            user['age'] = None

        # Deduplicate users based on name and feedback
        user_id = user.get('name', '').strip()
        if user_id not in unique_users:
            unique_users.add(user_id)
            clean_data.append(user)
    return clean_data

## Step 5: Analyse the Data

Extract key insights from the cleaned data:
- **Average Rating** — overall customer satisfaction score
- **Poor Rating Percentage** — proportion of users who rated below 3

In [42]:
clean_data(data)

[{'name': 'Alice', 'rating': 5, 'feedback': 'Great product!!', 'age': 25},
 {'name': 'Bob', 'rating': 4, 'feedback': 'ok but late Delivery', 'age': 30},
 {'name': ' Charlie', 'rating': 2, 'feedback': 'BAD EXPERIENCE ', 'age': None},
 {'name': 'Diana', 'feedback': 'Loved it!', 'rating': 5, 'age': 28},
 {'name': 'Eve',
  'rating': 3,
  'feedback': 'Average - could be better',
  'age': 20}]

In [50]:
# Get meaningful insights from the data
def analyze_data(data):
    # Average rating
    ratings = [user['rating'] for user in data if user['rating'] is not None]
    avg_rating = sum(ratings) / len(ratings) if ratings else 0 
    print(f"Average Rating: {avg_rating:.2f}")

    # Percentage of users gave poor rating (rating <3)
    poor_ratings = [user['rating'] for user in data if user['rating'] is not None and user['rating'] < 3]
    poor_rating_percentage = (len(poor_ratings) / len(ratings) * 100) if ratings else 0
    print(f"Percentage of Poor Ratings: {poor_rating_percentage:.2f}%")


In [51]:
analyze_data(clean_data(data))

Average Rating: 3.80
Percentage of Poor Ratings: 20.00%


## Step 6: Product Recommendations

Based on user ratings, recommend products:
- **Rating ≥ 4** → Recommend Apple (satisfied customers, upsell premium)
- **Rating < 4** → Recommend Samsung (dissatisfied customers, suggest alternative)

In [52]:
# Recommendation feature based on rating 
# If user rating >=4 recomend similar product say apple if user rating <4 recommend competitor say samsung
def recommend_product(data):
    recommendations = []
    for user in data:
        if user['rating'] is not None:
            if user['rating'] >= 4:
                recommendations.append((user['name'], 'Recommend Apple'))
            else:
                recommendations.append((user['name'], 'Recommend Samsung'))
    return recommendations

In [53]:
recommend_product(clean_data(data))

[('Alice', 'Recommend Apple'),
 ('Bob', 'Recommend Apple'),
 (' Charlie', 'Recommend Samsung'),
 ('Diana', 'Recommend Apple'),
 ('Eve', 'Recommend Samsung')]